In [2]:
# Drop the existing table first (safe)
spark.sql("DROP TABLE IF EXISTS silver_financial_statements")

# Now create clean Silver table
df_silver = spark.sql("""
SELECT 
    Year,
    TRIM(Company) AS Company,
    TRIM(Category) AS Category,
    Market_Cap_B_USD,
    Revenue,
    Gross_Profit,
    Net_Income,
    Earning_Per_Share AS EPS,
    EBITDA,
    Share_Holder_Equity AS Shareholder_Equity,
    Current_Ratio,
    Debt_Ratio,
    ROE,
    ROA,
    ROI,
    Net_Profit_Margin,
    Number_of_Employees,

    -- Clean percentages (max 2 decimals)
    ROUND(CASE WHEN Revenue > 0 THEN (Gross_Profit / Revenue) * 100 ELSE NULL END, 2) AS Gross_Margin_Pct,
    ROUND(CASE WHEN Revenue > 0 THEN (Net_Income / Revenue) * 100 ELSE NULL END, 2) AS Net_Profit_Margin_Pct,
    ROUND(CASE WHEN Share_Holder_Equity > 0 THEN (Net_Income / Share_Holder_Equity) * 100 ELSE NULL END, 2) AS ROE_Pct,

    -- YoY Growth
    ROUND(CASE WHEN LAG(Revenue) OVER (PARTITION BY Company ORDER BY Year) > 0 
         THEN ((Revenue - LAG(Revenue) OVER (PARTITION BY Company ORDER BY Year)) 
               / LAG(Revenue) OVER (PARTITION BY Company ORDER BY Year)) * 100 
         ELSE NULL END, 2) AS Revenue_Growth_YoY_Pct

FROM bronze_financial_statements
""")

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_financial_statements")

print("✅ Silver table created successfully with clean 2-decimal percentages!")
display(df_silver.limit(5))


StatementMeta(, 8a1e15b9-b37d-47cb-ab88-33df8f56ab7d, 4, Finished, Available, Finished, False)

✅ Silver table created successfully with clean 2-decimal percentages!


SynapseWidget(Synapse.DataFrame, 80c7cc04-7f81-4987-9489-5fc64eef4352)

In [4]:
# Drop the old Gold table first to avoid schema conflict
spark.sql("DROP TABLE IF EXISTS gold_company_yearly_kpis_table")

# Create fresh Gold table with all necessary columns for accurate KPIs
df_gold = spark.sql("""
SELECT 
    Year,
    Company,
    Category,
    Revenue,
    Net_Income,
    EPS,
    Market_Cap_B_USD,
    Shareholder_Equity,
    Gross_Margin_Pct,
    Net_Profit_Margin_Pct,
    ROE_Pct,
    Revenue_Growth_YoY_Pct,
    Current_Ratio,
    Debt_Ratio,
    Number_of_Employees,
    ROE,
    ROA,
    ROI
FROM silver_financial_statements
""")

df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_company_yearly_kpis_table")

print("✅ Gold table recreated successfully with all required columns!")
display(df_gold.limit(5))

StatementMeta(, 8a1e15b9-b37d-47cb-ab88-33df8f56ab7d, 6, Finished, Available, Finished, False)

✅ Gold table recreated successfully with all required columns!


SynapseWidget(Synapse.DataFrame, 849245e3-d6d7-4459-ade7-7817d3b31c7f)